# Chapter 20 — Build a Self-Improving Engineering Program

**Book alignment:** DSPy From First Principles, Chapter 20

**Question this notebook isolates:** Does the full repair loop fail the broken baseline on public and hidden validation while passing the control patch on both?


In [ ]:
from pathlib import Path
import ast
import importlib.util
import sys
import tempfile


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy
from common.fingerprints import fingerprint
from common.promotion import DeploymentManifest, PromotionRecord, ComparisonDecision
from common.repository_tools import RepositoryTools


class GeneratePatch(dspy.Signature):
    """Generate a bounded unified diff for a diagnosed issue."""

    issue: str = dspy.InputField()
    diagnosis: str = dspy.InputField()
    patch: str = dspy.OutputField()


patch_program = dspy.Predict(GeneratePatch)
HIDDEN = [
    ("hidden-basic-quantity", ((10.0, 2), (3.0, 4)), 32.0),
    ("hidden-single", ((7.5, 3),), 22.5),
    ("hidden-zero-quantity", ((8.0, 0), (2.0, 5)), 10.0),
    ("hidden-empty", (), 0.0),
]
print("dspy", dspy.__version__, "| patch program constructed, never executed; control patch is deterministic")


## The insufficient baseline fails in public

The fixture defect is concrete: `invoice_total` ignores quantity. Bounded diagnosis tools touch only the allowed files before any validation runs.


In [ ]:
_tmp = tempfile.TemporaryDirectory()
REPO = Path(_tmp.name) / "repair-checkout"
REPO.mkdir()
(REPO / "models.py").write_text("class Item:\n    def __init__(self, price, quantity):\n        self.price = price\n        self.quantity = quantity\n", encoding="utf-8")
(REPO / "service.py").write_text("from models import Item\n\n\ndef invoice_total(items):\n    return sum(item.price for item in items)\n", encoding="utf-8")

diag_tools = RepositoryTools(root=REPO)
tool_calls = [diag_tools.search_repository("invoice_total"), diag_tools.read_file("service.py", 1, 20), diag_tools.inspect_symbol("invoice_total")]
print("diagnosis tool calls:", len(tool_calls), "| all ok:", all(t.ok for t in tool_calls))

spec_mod = importlib.util.spec_from_file_location("broken_service", str(REPO / "service.py"))
broken = importlib.util.module_from_spec(spec_mod)
sys.modules["broken_service"] = broken
spec_mod.loader.exec_module(broken)

class Priced:
    def __init__(self, price, quantity):
        self.price, self.quantity = price, quantity

public_items = [Priced(10.0, 2), Priced(3.0, 4)]
broken_public = broken.invoice_total(public_items)
print("broken public total:", broken_public, "| expected: 32.0")


In [ ]:
assert all(t.ok for t in tool_calls)
assert broken_public != 32.0
print("public validation fails on the broken baseline, independently of any model claim")


## The control patch passes public and hidden validation

A known deterministic repair proves the harness can recognize the intended fix. Scope stays in `service.py`, syntax parses, and hidden evaluation sees only the frozen candidate.


In [ ]:
FIXED_SERVICE = "from models import Item\n\n\ndef invoice_total(items):\n    return sum(item.price * item.quantity for item in items)\n"
ast.parse(FIXED_SERVICE)
fixed_path = REPO / "service.py"
fixed_path.write_text(FIXED_SERVICE, encoding="utf-8")
spec_fixed = importlib.util.spec_from_file_location("fixed_service", str(fixed_path))
fixed = importlib.util.module_from_spec(spec_fixed)
sys.modules["fixed_service"] = fixed
spec_fixed.loader.exec_module(fixed)

fixed_public = fixed.invoice_total(public_items)
hidden_pass = sum(1 for _, lines, want in HIDDEN if abs(fixed.invoice_total([Priced(p, q) for p, q in lines]) - want) < 1e-9)
broken_pass = sum(1 for _, lines, want in HIDDEN if abs(broken.invoice_total([Priced(p, q) for p, q in lines]) - want) < 1e-9)
print(f"fixed public: {fixed_public} | hidden: {hidden_pass}/{len(HIDDEN)} -> 1.0")
print(f"broken hidden: {broken_pass}/{len(HIDDEN)} -> {broken_pass / len(HIDDEN):.1f}")


In [ ]:
assert fixed_public == 32.0
assert hidden_pass == 4
assert broken_pass == 2 and (broken_pass / len(HIDDEN)) == 0.5
print("candidate 1.0 vs broken 0.5 on independent hidden cases; no revision needed")


## Manifest guard and one-way evidence

The candidate cannot be interpreted if the manifest is incomplete or the holdout boundary is broken, and the promotion decision is a separate fingerprinted object from the candidate artifact.


In [ ]:
def validate_experiment_manifest(manifest):
    required = {"program_version", "dataset_fingerprint", "train_ids", "dev_ids", "holdout_ids"}
    missing = sorted(required - set(manifest))
    if missing:
        raise ValueError(f"manifest missing: {', '.join(missing)}")
    visible = set(manifest["train_ids"]) | set(manifest["dev_ids"])
    leaked = visible & set(manifest["holdout_ids"])
    if leaked:
        raise ValueError(f"holdout ids optimizer-visible: {', '.join(sorted(leaked))}")

manifest = {
    "program_version": "repair-v2", "dataset_fingerprint": fingerprint(HIDDEN),
    "train_ids": ["repair-train-1"], "dev_ids": ["repair-dev-1"], "holdout_ids": ["hidden-basic-quantity"],
}
validate_experiment_manifest(manifest)
leak_error = None
try:
    validate_experiment_manifest({**manifest, "dev_ids": ["hidden-basic-quantity"]})
except ValueError as exc:
    leak_error = str(exc)
candidate_fp = fingerprint({"program": "repair-v2", "patch": FIXED_SERVICE})
decision = PromotionRecord(ComparisonDecision.PROMOTE, "hidden 1.0 without regression", "repair-v1", "repair-v2")
print("clean manifest passes; overlap rejected:", leak_error)
print("candidate fp:", candidate_fp[:12], "| decision fp:", decision.to_record()["fingerprint"][:12])


In [ ]:
assert leak_error is not None and "optimizer-visible" in leak_error
assert candidate_fp != decision.to_record()["fingerprint"]
assert decision.decision == ComparisonDecision.PROMOTE
print("manifest explains under which boundary the candidate was produced; activation stays manual")


## What we earned

The mechanisms compose: contract, bounded tools, validation loop, one-way evidence, manifest guard, and promotion record each held, and the model was never allowed to certify its own success.

Notebook 21 / Chapter 21 (Appendix) shows three of these mechanisms already running in a production codebase, with only the deterministic mechanics claimed as measured.
